# Search Service 실험

Domain Agent가 사용하는 `SearchService`를 직접 호출해 검색 결과를 확인한다.

준비 사항:

- 저장소 루트의 `.env`에 `CLOVASTUDIO_*`, `QDRANT_*` 설정하기
- 대상 Qdrant collection에 문서 인덱스가 적재되어 있는지 확인하기
- 저장소 루트에서 `uv run --locked --group notebook jupyter lab` 실행하기

검색 원문이 출력에 남을 수 있으므로 커밋 전에 출력을 모두 지운다.

In [ ]:
# 검색 요청과 도메인별 문서 권한
from pension_agent.agent.contracts import Permission
from pension_agent.agent.search import SearchRequest, SearchService

# CLOVA embedding과 Qdrant 연결 설정
from pension_agent.config import (
    BGE_M3_EMBEDDING_CONFIG,
    ClovaStudioConnection,
    QdrantConnection,
)
from pension_agent.retrieval import (
    create_async_qdrant_client,
    create_async_qdrant_retriever,
    create_clova_query_embedder,
)

## 연결과 Search Service 조립

`.env`의 설정으로 embedding과 Qdrant 검색기를 준비한다.

In [ ]:
# 셀을 다시 실행하면 기존 Qdrant 연결을 먼저 닫는다.
existing_qdrant_client = globals().get("qdrant_client")
if existing_qdrant_client is not None:
    await existing_qdrant_client.close()

# 저장소 루트의 .env에서 연결 정보를 읽는다.
clova_connection = ClovaStudioConnection()
qdrant_connection = QdrantConnection()

# 질의 embedding과 Qdrant 조회기를 Search Service에 연결한다.
embedder = create_clova_query_embedder(
    config=BGE_M3_EMBEDDING_CONFIG,
    connection=clova_connection,
)
qdrant_client = create_async_qdrant_client(qdrant_connection)
retriever = create_async_qdrant_retriever(
    qdrant_client,
    connection=qdrant_connection,
)
search_service = SearchService(embedder=embedder, retriever=retriever)

# API key를 제외한 연결 대상만 확인한다.
{
    "qdrant_url": qdrant_connection.url,
    "collection": qdrant_connection.collection,
    "embedding_model": BGE_M3_EMBEDDING_CONFIG.model,
}

## 전체 문서 검색

파일을 지정하지 않으면 도메인 권한 범위의 전체 문서를 검색한다.

In [ ]:
# POLICY는 연금 참고 문서만 검색한다.
request = SearchRequest(
    objective="IRP 계좌 이전 절차와 필요한 조건",
    # 가장 관련 있는 청크의 앞뒤 문맥도 가져온다.
    expand_neighbors=True,
)
result = await search_service.search(
    request,
    permission=Permission.POLICY,
)
result.model_dump()

## 문서 범위를 제한한 검색

파일명을 주면 해당 문서 안에서만 검색한다.

In [ ]:
# Qdrant에 저장된 source_file_name으로 바꾸어 실행한다.
request = SearchRequest(
    objective="연금 수령 시 세율",
    source_file_name="원본-파일명.pdf",
)
result = await search_service.search(
    request,
    permission=Permission.TAX_PAYOUT,
)
result.model_dump()

## 정리

실험을 마치면 Qdrant 연결을 닫는다.

In [ ]:
# 외부 연결을 정리한다.
await qdrant_client.close()